# KLE Colab — XGBoost + Multi-Strategy Voting

目标：**10+ hits / 20**

## 核心策略

| 投票者 | 逻辑 | 特点 |
|--------|------|------|
| **XGBoost ×80** | 每个号码一个二分类器，基于丰富特征 | 表格数据之王，训练极快 |
| Frequency | 近 N 期出现频率 | 稳定 baseline |
| Gap | 遗漏回补（越久未出分越高） | 捕捉回归均值 |
| Momentum | 近期连续出现 | 捕捉短期趋势 |

## XGBoost 特征（每个号码独立）
- 近 10/20/50 期出现频率
- 距上次出现间隔 (gap)
- 连续出现/缺席轮数 (streak)
- 所在区间(1-20/21-40/41-60/61-80)的整体活跃度
- 奇偶、大小比例
- 尾数热度

## 为什么比 CNN 好
1. CNN 假设号码有"空间结构"（8×10 图像）→ 假设不成立
2. XGBoost 直接学习"号码 X 在这些统计条件下出现概率"→ 假设合理
3. XGBoost 训练 80 个分类器 ~2s，CNN 一个模型 ~10s

建议 Colab 运行时切换到 `GPU`（XGBoost 也支持 GPU 加速）。

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def run(cmd: str) -> None:
    print(f"+ {cmd}")
    subprocess.run(cmd, shell=True, check=True)


REPO_URL = "https://github.com/jursmatsko/lotto-image-predictor-research.git"
REPO_ROOT = Path("/content/lotto-image-predictor-research")
PROJECT_DIR = REPO_ROOT / "kle"

run("python -m pip install -q pandas matplotlib requests beautifulsoup4 lxml xgboost")

try:
    import torch
except ImportError:
    run("python -m pip install -q torch")
    import torch

if not REPO_ROOT.exists():
    run(f"git clone {REPO_URL} {REPO_ROOT}")
else:
    print(f"Repo already exists: {REPO_ROOT}")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Python: {sys.version.split()[0]}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

+ python -m pip install -q pandas matplotlib requests beautifulsoup4 lxml xgboost


KeyboardInterrupt: 

## 数据检查

先确认 `data/data.csv` 能正确读取，并检查最近几期数据结构。

In [ ]:
import pandas as pd


data_path = PROJECT_DIR / "data" / "data.csv"
df = pd.read_csv(data_path)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
display(df.head())
display(df.tail())

FileNotFoundError: [Errno 2] No such file or directory: '/content/lotto-image-predictor-research/kle/data/data.csv'

## 加载应用

这里直接复用仓库里的 `ImagePredictionApp`，但训练模型选择 `pytorch_cnn`，因为它在 Colab 的兼容性和速度都更好。

In [ ]:
from image_predictor.main import ImagePredictionApp
from image_predictor.models.pytorch_image_cnn import DEVICE


app = ImagePredictionApp()
app.loader.load_data()

print(f"Training device: {DEVICE}")
print(app.loader.get_statistics())

## 参数配置

| 参数 | 作用 |
|------|------|
| `WF_EVAL_LAST` | 在最近多少期上做 walk-forward 回测 |
| `TRAIN_WINDOW` | 训练窗口大小（最近 N 期数据用于训练） |
| `XGB_PARAMS` | XGBoost 超参数（树深、学习率、树数量等） |
| `W_XGB` | XGBoost 模型在融合中的权重 |
| `W_FREQUENCY / W_GAP / W_MOMENTUM` | 统计策略在融合中的权重 |

**特征工程**（14 维 × 80 个号码）：多窗口频率 + 遗漏间隔 + 连中/连缺 + 区间活跃度 + 尾数热度 + 静态属性。

XGBoost 对表格数据效果极佳，训练速度比 CNN 快 ~10 倍。

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Walk-forward 配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WF_EVAL_LAST = 20
TRAIN_WINDOW = 300

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  XGBoost 超参数（增强正则化防止过拟合噪声）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
XGB_PARAMS = dict(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=300,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=5,
    reg_alpha=0.5,
    reg_lambda=2.0,
    gamma=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    verbosity=0,
    random_state=42,
)

SAMPLE_DECAY = 0.005   # 近期训练样本权重更大

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  多策略融合权重
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
W_XGB        = 0.40
W_FREQUENCY  = 0.30
W_GAP        = 0.15
W_MOMENTUM   = 0.15

FREQ_WINDOW     = 30
GAP_WINDOW      = 100
MOMENTUM_WINDOW = 5

NUM_TICKETS = 10

MODEL_CACHE_DIR = PROJECT_DIR / "image_predictor" / "models" / "saved"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{'─'*56}")
print(f"  XGBoost     : depth={XGB_PARAMS['max_depth']}  trees={XGB_PARAMS['n_estimators']}  lr={XGB_PARAMS['learning_rate']}")
print(f"  Regularize  : alpha={XGB_PARAMS['reg_alpha']} lambda={XGB_PARAMS['reg_lambda']} gamma={XGB_PARAMS['gamma']}")
print(f"  Sample decay: {SAMPLE_DECAY}")
print(f"  Train window: {TRAIN_WINDOW}  |  WF eval last: {WF_EVAL_LAST}")
print(f"  Fusion      : XGB={W_XGB} Freq={W_FREQUENCY} Gap={W_GAP} Mom={W_MOMENTUM}")
print(f"  Features    : 20 dims (expanded)")
print(f"{'─'*56}")

In [ ]:
import time
import numpy as np
from tqdm.notebook import tqdm
import xgboost as xgb

# ═══════════════════════════════════════════════════════
#  Feature Engineering — 20 features per number
# ═══════════════════════════════════════════════════════

FEAT_WINDOWS = [5, 10, 20, 50]
EXP_LAMBDAS  = [0.1, 0.05]
N_FEATURES   = 20

_ZONE_ID     = np.array([n // 20 for n in range(80)], dtype=np.float32) / 3.0
_TAIL_DIG    = np.array([n % 10 for n in range(80)], dtype=np.float32) / 9.0
_IS_ODD      = np.array([(n + 1) % 2 for n in range(80)], dtype=np.float32)
_IS_HIGH     = np.array([1.0 if n >= 40 else 0.0 for n in range(80)], dtype=np.float32)
_POS_NORM    = np.arange(1, 81, dtype=np.float32) / 80.0
_TAIL_MASK   = np.array([[n % 10 == d for n in range(80)] for d in range(10)])
_ZONE_RANGES = [(z * 20, (z + 1) * 20) for z in range(4)]

FEAT_NAMES = [
    "freq_5", "freq_10", "freq_20", "freq_50",
    "exp_freq_.1", "exp_freq_.05",
    "gap", "hit_streak", "miss_streak",
    "hot_3of5", "cold_0of10", "overall_freq",
    "zone_id", "zone_act_10", "zone_dev",
    "tail_dig", "tail_act_10",
    "is_odd", "is_high", "pos_norm",
]


def compute_features_at_t(presence, t):
    """Feature matrix for all 80 numbers at draw index t.  Shape (80, 20)."""
    feats = []

    # 1-4: multi-window frequency
    for w in FEAT_WINDOWS:
        s = max(0, t - w)
        feats.append(presence[s:t].sum(axis=0) / max(t - s, 1))

    # 5-6: exponentially weighted frequency
    for lam in EXP_LAMBDAS:
        lb = min(t, 100)
        if lb > 0:
            w = np.exp(-lam * np.arange(lb)[::-1])
            feats.append((presence[t - lb:t] * w[:, None]).sum(axis=0) / w.sum())
        else:
            feats.append(np.zeros(80, dtype=np.float32))

    # 7: gap since last appearance
    gaps = np.full(80, min(t, GAP_WINDOW), dtype=np.float32)
    found = np.zeros(80, dtype=bool)
    for i in range(t - 1, max(t - GAP_WINDOW - 1, -1), -1):
        newly = (presence[i] > 0) & ~found
        gaps[newly] = t - 1 - i
        found |= presence[i] > 0
        if found.all():
            break
    feats.append(gaps / GAP_WINDOW)

    # 8: consecutive-hit streak
    hs = np.zeros(80, dtype=np.float32)
    active = np.ones(80, dtype=bool)
    for i in range(t - 1, max(t - 30, -1), -1):
        active &= presence[i] > 0
        if not active.any():
            break
        hs[active] += 1
    feats.append(hs / 10.0)

    # 9: consecutive-miss streak
    ms = np.zeros(80, dtype=np.float32)
    active = np.ones(80, dtype=bool)
    for i in range(t - 1, max(t - 30, -1), -1):
        active &= presence[i] == 0
        if not active.any():
            break
        ms[active] += 1
    feats.append(ms / 10.0)

    # 10: hot indicator (3+ in last 5)
    s5 = max(0, t - 5)
    feats.append((presence[s5:t].sum(axis=0) >= 3).astype(np.float32))

    # 11: cold indicator (0 in last 10)
    s10 = max(0, t - 10)
    feats.append((presence[s10:t].sum(axis=0) == 0).astype(np.float32))

    # 12: overall historical frequency
    feats.append(presence[:t].sum(axis=0) / max(t, 1))

    # 13: zone id (static)
    feats.append(_ZONE_ID)

    # 14: zone activity in last 10
    w10 = t - s10
    zone_act = np.zeros(80, dtype=np.float32)
    if w10 > 0:
        chunk10 = presence[s10:t].sum(axis=0)
        for a, b in _ZONE_RANGES:
            zone_act[a:b] = chunk10[a:b].sum() / (20 * w10)
    feats.append(zone_act)

    # 15: zone deviation from expected 25%
    feats.append(zone_act - 0.25)

    # 16: tail digit (static)
    feats.append(_TAIL_DIG)

    # 17: tail-digit activity in last 10
    tail_act = np.zeros(80, dtype=np.float32)
    if w10 > 0:
        for d in range(10):
            mask = _TAIL_MASK[d]
            tail_act[mask] = chunk10[mask].sum() / (8 * w10)
    feats.append(tail_act)

    # 18-20: static
    feats.append(_IS_ODD)
    feats.append(_IS_HIGH)
    feats.append(_POS_NORM)

    return np.stack(feats, axis=1)


def build_training_data(presence, w_start, end_t, decay=0.0):
    """Build (X, y, sample_weight) for XGBoost."""
    min_t = max(w_start, max(FEAT_WINDOWS) + 1)
    n = end_t - min_t
    X = np.empty((n * 80, N_FEATURES), dtype=np.float32)
    y = np.empty(n * 80, dtype=np.float32)
    sw = np.empty(n * 80, dtype=np.float32)
    for idx, t in enumerate(range(min_t, end_t)):
        X[idx * 80:(idx + 1) * 80] = compute_features_at_t(presence, t)
        y[idx * 80:(idx + 1) * 80] = presence[t]
        sw[idx * 80:(idx + 1) * 80] = np.exp(decay * (t - (end_t - 1)))
    return X, y, sw


# ═══════════════════════════════════════════════════════
#  Statistical strategies (for fusion)
# ═══════════════════════════════════════════════════════

def stat_frequency_scores(draws, window):
    recent = draws[-window:]
    counts = np.zeros(80, dtype=np.float32)
    for draw in recent:
        for n in draw:
            counts[n - 1] += 1
    mx = counts.max()
    return counts / mx if mx > 0 else counts


def stat_gap_scores(draws, window):
    recent = draws[-window:]
    last_seen = np.full(80, -1, dtype=np.float32)
    for i, draw in enumerate(recent):
        for n in draw:
            last_seen[n - 1] = i
    total = len(recent)
    gaps = np.where(last_seen >= 0, total - 1 - last_seen, total).astype(np.float32)
    mx = gaps.max()
    return gaps / mx if mx > 0 else gaps


def stat_momentum_scores(draws, window):
    recent = draws[-window:]
    scores = np.zeros(80, dtype=np.float32)
    for t, draw in enumerate(recent):
        w = (t + 1) / len(recent)
        for n in draw:
            scores[n - 1] += w
    mx = scores.max()
    return scores / mx if mx > 0 else scores


# ═══════════════════════════════════════════════════════
#  Prepare data
# ═══════════════════════════════════════════════════════
app.loader.load_data()
all_draws = app.loader.draws
issues    = app.loader.issues

presence = np.zeros((len(all_draws), 80), dtype=np.float32)
for i, d in enumerate(all_draws):
    for n in d:
        presence[i, n - 1] = 1.0

eval_start = max(max(FEAT_WINDOWS) + 1, len(all_draws) - WF_EVAL_LAST)
n_steps    = len(all_draws) - eval_start

print(f"Total draws       : {len(all_draws)}")
print(f"Walk-forward steps: {n_steps}")
print(f"Features/number   : {N_FEATURES}")
print(f"Training samples  : ~{TRAIN_WINDOW * 80:,}")
print(f"Sample decay      : {SAMPLE_DECAY}")
print(f"Fusion            : XGB={W_XGB} Freq={W_FREQUENCY} Gap={W_GAP} Mom={W_MOMENTUM}")
print("=" * 60)

# ═══════════════════════════════════════════════════════
#  Walk-forward backtest
# ═══════════════════════════════════════════════════════
wf_results   = []
running_hits = []
wf_start     = time.time()

strat_hits = {"xgb": [], "freq": [], "gap": [], "mom": [], "combined": []}

step_bar = tqdm(range(eval_start, len(all_draws)), desc="Walk-forward", unit="draw")

for test_idx in step_bar:
    t0 = time.time()
    w_start = max(0, test_idx - TRAIN_WINDOW) if TRAIN_WINDOW else 0

    # ── XGBoost ────────────────────────────────────────
    X_train, y_train, sw = build_training_data(presence, w_start, test_idx, SAMPLE_DECAY)
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X_train, y_train, sample_weight=sw)

    X_test = compute_features_at_t(presence, test_idx)
    xgb_probs = model.predict_proba(X_test)[:, 1]
    xgb_norm = xgb_probs / xgb_probs.max() if xgb_probs.max() > 0 else xgb_probs

    # ── Statistical strategies ─────────────────────────
    history = all_draws[:test_idx]
    freq_sc = stat_frequency_scores(history, FREQ_WINDOW)
    gap_sc  = stat_gap_scores(history, GAP_WINDOW)
    mom_sc  = stat_momentum_scores(history, MOMENTUM_WINDOW)

    # ── Fuse → top 20 ─────────────────────────────────
    combined = (W_XGB * xgb_norm + W_FREQUENCY * freq_sc +
                W_GAP * gap_sc   + W_MOMENTUM * mom_sc)
    top20_idx = np.argsort(combined)[::-1][:20]
    pred_nums = sorted([int(i + 1) for i in top20_idx])

    pred_set   = set(pred_nums)
    actual_set = set(all_draws[test_idx])
    hits = len(pred_set & actual_set)

    # per-strategy comparison
    for name, sc in [("xgb", xgb_norm), ("freq", freq_sc), ("gap", gap_sc), ("mom", mom_sc)]:
        s_pred = set(int(i + 1) for i in np.argsort(sc)[::-1][:20])
        strat_hits[name].append(len(s_pred & actual_set))
    strat_hits["combined"].append(hits)

    running_hits.append(hits)
    avg_so_far = np.mean(running_hits)
    step_time  = time.time() - t0

    wf_results.append({
        "issue":     issues[test_idx],
        "hits":      hits,
        "predicted": sorted(pred_set),
        "actual":    sorted(actual_set),
    })
    step_bar.set_postfix(
        hits=f"{hits}/20", avg=f"{avg_so_far:.2f}",
        vs_rand=f"{avg_so_far - 5:+.2f}", t=f"{step_time:.0f}s",
    )

    step_n = len(running_hits)
    mark = "🟢" if hits >= 10 else "🔵" if hits > 5 else "🔴"
    pred_str  = " ".join(f"{n:02d}" for n in sorted(pred_set))
    act_str   = " ".join(f"{n:02d}" for n in sorted(actual_set))
    match_str = " ".join(f"{n:02d}" for n in sorted(pred_set & actual_set))
    p_min, p_max, p_std = xgb_probs.min(), xgb_probs.max(), xgb_probs.std()
    print(f"\n{mark} [{step_n:>2}/{n_steps}] Draw {issues[test_idx]}  "
          f"hits={hits:>2}/20  avg={avg_so_far:.2f}  ({step_time:.0f}s)")
    print(f"   Predicted : {pred_str}")
    print(f"   Actual    : {act_str}")
    print(f"   Matched   : {match_str if match_str else '(none)'}")
    print(f"   XGB probs : min={p_min:.4f}  max={p_max:.4f}  std={p_std:.4f}")

elapsed   = time.time() - wf_start
avg_final = np.mean(running_hits)
above_10  = sum(h >= 10 for h in running_hits)
print(f"\n{'='*60}")
print(f"  Walk-forward complete in {elapsed/60:.1f} min")
print(f"  Average hits   : {avg_final:.2f} / 20")
print(f"  vs Random      : {avg_final - 5:+.2f}")
print(f"  Draws >= 10    : {above_10} / {len(running_hits)}")
print(f"  Draws > 5      : {sum(h > 5 for h in running_hits)} / {len(running_hits)}")
print(f"{'='*60}")
print(f"\n  Strategy comparison (each alone):")
for name in ["xgb", "freq", "gap", "mom", "combined"]:
    a = np.mean(strat_hits[name])
    print(f"    {name:10s}: {a:.2f} avg hits  (vs_rand {a-5:+.2f})")
print(f"    {'random':10s}: 5.00")
print(f"{'='*60}")

## Walk-Forward 回测结果

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

wf_df = pd.DataFrame(wf_results)
avg   = wf_df["hits"].mean()
above = (wf_df["hits"] > 5).sum()

print("=" * 50)
print(f"  Average hits  : {avg:.2f} / 20")
print(f"  Random baseline: 5.00 / 20")
print(f"  vs Random      : {avg - 5:+.2f}")
print(f"  Draws > 5 hits : {above} / {len(wf_df)}")
print("=" * 50)
display(wf_df[["issue", "hits"]])

# ── 柱状图 ───────────────────────────────────────────────
colors = ["#2ecc71" if h > 5 else "#e74c3c" if h < 5 else "#f39c12"
          for h in wf_df["hits"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(len(wf_df)), wf_df["hits"], color=colors)
axes[0].axhline(5.0, color="orange", linestyle="--", linewidth=1.5, label="Random (5.0)")
axes[0].axhline(avg, color="royalblue", linestyle="-", linewidth=2, label=f"Average ({avg:.2f})")
axes[0].set_xlabel("Draw index")
axes[0].set_ylabel("Hits / 20")
axes[0].set_title("Walk-Forward Ensemble: Hits per Draw")
axes[0].legend()

green_p = mpatches.Patch(color="#2ecc71", label="> 5 hits")
red_p   = mpatches.Patch(color="#e74c3c", label="< 5 hits")
axes[0].legend(handles=[green_p, red_p,
    plt.Line2D([], [], color="orange", linestyle="--", label="Random"),
    plt.Line2D([], [], color="royalblue", label=f"Avg {avg:.2f}")
])

# 命中分布直方图
axes[1].hist(wf_df["hits"], bins=range(0, 21), color="#3498db", edgecolor="white", align="left")
axes[1].axvline(5.0, color="orange", linestyle="--", linewidth=1.5, label="Random (5.0)")
axes[1].axvline(avg,  color="royalblue", linestyle="-",  linewidth=2,   label=f"Avg ({avg:.2f})")
axes[1].set_xlabel("Hits")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Hit Distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig(str(MODEL_CACHE_DIR / "wf_results.png"), dpi=120, bbox_inches="tight")
plt.show()

## 最终预测 — 用全量数据训练 Ensemble

回测只用来评估策略质量。最终预测用**全量历史数据**重新训练一批 ensemble 模型，集成后给出下一期推荐。

In [ ]:
# ── 最终预测：XGBoost + 多策略融合 ──────────────────────
import matplotlib.pyplot as plt

T = len(all_draws)
w_start = max(0, T - TRAIN_WINDOW) if TRAIN_WINDOW else 0
print(f"Training final XGBoost — window: draws [{w_start}, {T})  ({T - w_start} draws)")

X_final, y_final, sw_final = build_training_data(presence, w_start, T, SAMPLE_DECAY)
final_model = xgb.XGBClassifier(**XGB_PARAMS)
final_model.fit(X_final, y_final, sample_weight=sw_final)
print(f"  ✓ Trained on {len(X_final):,} samples")

X_next = compute_features_at_t(presence, T)
xgb_probs = final_model.predict_proba(X_next)[:, 1]
xgb_norm = xgb_probs / xgb_probs.max() if xgb_probs.max() > 0 else xgb_probs

freq_final = stat_frequency_scores(all_draws, FREQ_WINDOW)
gap_final  = stat_gap_scores(all_draws, GAP_WINDOW)
mom_final  = stat_momentum_scores(all_draws, MOMENTUM_WINDOW)

combined = (W_XGB * xgb_norm + W_FREQUENCY * freq_final +
            W_GAP * gap_final + W_MOMENTUM * mom_final)
top20_idx = np.argsort(combined)[::-1][:20]
top20 = sorted([int(i + 1) for i in top20_idx])

score_map = combined.reshape(8, 10)
print(f"\n{'='*60}")
print(f"  Fusion: XGB={W_XGB} Freq={W_FREQUENCY} Gap={W_GAP} Mom={W_MOMENTUM}")
print(f"  Top 20: {top20}")
print(f"  XGB prob range: {xgb_probs.min():.4f} — {xgb_probs.max():.4f}  std={xgb_probs.std():.4f}")
print(f"{'='*60}")

importances = final_model.feature_importances_

# ── Visualization ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].imshow(score_map, cmap="hot", vmin=0, aspect="equal")
fig.colorbar(im0, ax=axes[0], label="Combined Score")
for r in range(8):
    for c in range(10):
        num = r * 10 + c + 1
        is_top = num in set(top20)
        axes[0].text(c, r, str(num), ha="center", va="center", fontsize=7,
                     fontweight="bold",
                     color="cyan" if is_top else
                     ("white" if score_map[r, c] > score_map.mean() else "gray"))
axes[0].set_title("Fused Score Map — Top20 = cyan")
axes[0].set_xticks(range(10)); axes[0].set_xticklabels(range(1, 11))
axes[0].set_yticks(range(8))
axes[0].set_yticklabels([f"{r*10+1}-{r*10+10}" for r in range(8)])

xgb_map = xgb_norm.reshape(8, 10)
im1 = axes[1].imshow(xgb_map, cmap="hot", vmin=0, aspect="equal")
fig.colorbar(im1, ax=axes[1], label="XGBoost Prob")
axes[1].set_title("XGBoost Only")
axes[1].set_xticks(range(10)); axes[1].set_xticklabels(range(1, 11))
axes[1].set_yticks(range(8))
axes[1].set_yticklabels([f"{r*10+1}-{r*10+10}" for r in range(8)])

sorted_idx = np.argsort(importances)[::-1]
axes[2].barh(range(len(FEAT_NAMES)),
             importances[sorted_idx], color="#3498db")
axes[2].set_yticks(range(len(FEAT_NAMES)))
axes[2].set_yticklabels([FEAT_NAMES[i] for i in sorted_idx])
axes[2].set_xlabel("Importance")
axes[2].set_title("XGBoost Feature Importance")
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig(str(MODEL_CACHE_DIR / "ensemble_heatmap.png"), dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
ranked = np.argsort(combined)[::-1]
top_pool = ranked[:25]
pool_weights = combined[top_pool].copy()
pool_weights /= pool_weights.sum()

tickets = []
tickets.append(sorted([int(ranked[i] + 1) for i in range(10)]))

for seed in range(1, NUM_TICKETS):
    rng = np.random.RandomState(seed * 7 + 13)
    w = pool_weights.copy()
    chosen = []
    for _ in range(10):
        w_norm = w / w.sum()
        pick = rng.choice(len(top_pool), p=w_norm)
        chosen.append(int(top_pool[pick] + 1))
        w[pick] = 0.0
    tickets.append(sorted(chosen))

avg = np.mean([r["hits"] for r in wf_results])

print(f"{'='*50}")
print(f"  Next Draw Prediction")
print(f"  Top 20: {' '.join(f'{n:02d}' for n in top20)}")
print(f"{'='*50}")
print(f"\n  {NUM_TICKETS} Recommended Tickets (10 numbers each):")
for i, ticket in enumerate(tickets, 1):
    print(f"  Ticket {i:02d}: {' '.join(f'{n:02d}' for n in ticket)}")

print(f"\n  Walk-Forward Backtest Summary ({WF_EVAL_LAST} draws):")
print(f"  Average Hits : {avg:.2f}/20  (Random baseline: 5.00/20)")
print(f"  vs Random    : {avg - 5:+.2f}")
print(f"\n  Lottery is random. This is for scientific research only.")

## 可选：保存预测结果到 Google Drive

Colab 会话结束后本地文件会消失。运行下面这个单元，把图表和预测结果保存到 Google Drive 长期保留。

In [ ]:
import shutil, json
from google.colab import drive

drive.mount('/content/drive')

drive_dir = Path('/content/drive/MyDrive/kle_results')
drive_dir.mkdir(parents=True, exist_ok=True)

# 保存图表
for fname in ["wf_results.png", "ensemble_heatmap.png"]:
    src = MODEL_CACHE_DIR / fname
    if src.exists():
        shutil.copy(src, drive_dir / fname)
        print(f"✓ Saved chart : {fname}")

# 保存预测结果 JSON
result = {
    "model": "xgboost",
    "top20": top20,
    "tickets": tickets,
    "wf_avg_hits": round(float(avg), 3),
    "wf_vs_random": round(float(avg) - 5.0, 3),
    "wf_eval_last": WF_EVAL_LAST,
    "wf_details": wf_results,
}
result_path = drive_dir / "prediction_result.json"
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✓ Saved result: prediction_result.json")
print(f"\nAll files saved to: {drive_dir}")